# Module 4: Setup or Resume Environment

## Overview

This notebook helps you prepare for Module 4 (Troubleshooting & Incident Response). It validates that your LangSmith environment is running and accessible, or directs you to deploy it using Module 1.

**Prerequisites:**
- Module 1 notebooks available (for deployment if needed)
- kubectl configured (if environment exists)
- Cloud provider credentials (if deploying)

**What This Notebook Does:**
1. Checks if LangSmith is already deployed
2. If not, provides links to Module 1 deployment notebooks
3. If yes, validates the environment is healthy and reachable
4. Confirms prerequisites for Module 4 failure labs

**Estimated time:** 10-15 minutes


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
possible_paths = [
    Path.cwd().parent,  # If cwd is module-4, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## 1. Configuration

Load and validate configuration from environment variables.


In [ ]:
import os
from shared._validation import require_env, ok, warn
from shared._cloud_helpers import get_cloud_provider, get_region

# Required configuration
required_vars = ["NAMESPACE", "CLUSTER_NAME"]

print("### Loading Configuration\n")

config = {}
missing = []

for var in required_vars:
    value = os.environ.get(var, "").strip()
    if not value:
        missing.append(var)
    config[var] = value

if missing:
    raise RuntimeError(f"❌ Missing required environment variables: {', '.join(missing)}\n"
                      f"💡 Copy env-samples/workshop.env.example to your .env file and fill in values")

# Optional but recommended
config["HELM_RELEASE"] = os.environ.get("HELM_RELEASE", "langsmith")
config["LANGSMITH_DOMAIN"] = os.environ.get("LANGSMITH_DOMAIN", "")

# Show cloud provider info
provider = get_cloud_provider()
region = get_region()

print(f"Cloud Provider: {provider.upper()}")
print(f"Region: {region}")
print(f"Namespace: {config['NAMESPACE']}")
print(f"Cluster: {config['CLUSTER_NAME']}")
print(f"Helm Release: {config['HELM_RELEASE']}")

if config["LANGSMITH_DOMAIN"]:
    print(f"LangSmith Domain: {config['LANGSMITH_DOMAIN']}")

ok("Configuration loaded")


## 2. Check if Environment Exists

We'll check if LangSmith is already deployed. If not, we'll provide instructions to deploy using Module 1.


In [ ]:
from shared._shell import run
from shared._cloud_helpers import cluster_exists, configure_kubectl, get_kubernetes_service_name

namespace = config["NAMESPACE"]
cluster_name = config["CLUSTER_NAME"]
k8s_service = get_kubernetes_service_name()

print(f"### Checking {k8s_service} Cluster\n")

# Check if cluster exists
if cluster_exists(cluster_name):
    ok(f"Cluster '{cluster_name}' exists")
    
    # Configure kubectl
    print(f"\n### Configuring kubectl\n")
    try:
        configure_kubectl(cluster_name, region)
        ok("kubectl configured")
    except Exception as e:
        warn(f"Could not configure kubectl: {e}")
        print("💡 Make sure you have proper cloud provider credentials")
        raise
else:
    warn(f"Cluster '{cluster_name}' not found")
    print("\n💡 You need to deploy LangSmith first using Module 1 notebooks.")
    print("   See the 'Deploy Environment' section below.")
    raise RuntimeError("Cluster not found. Deploy using Module 1 first.")


## 3. Verify Namespace and Helm Release

Check that the LangSmith namespace exists and Helm release is installed.


In [ ]:
import json

helm_release = config["HELM_RELEASE"]

print("### Checking Namespace\n")
result = run(
    ["kubectl", "get", "namespace", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    ok(f"Namespace '{namespace}' exists")
else:
    warn(f"Namespace '{namespace}' not found")
    print("\n💡 You need to deploy LangSmith first using Module 1 notebooks.")
    print("   See the 'Deploy Environment' section below.")
    raise RuntimeError("Namespace not found. Deploy using Module 1 first.")

print("\n### Checking Helm Release\n")
result = run(
    ["helm", "list", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    releases = json.loads(result.stdout)
    release_found = any(r.get("name") == helm_release for r in releases)
    
    if release_found:
        ok(f"Helm release '{helm_release}' found")
        # Get release info
        result = run(
            ["helm", "status", helm_release, "-n", namespace, "-o", "json"],
            check=False,
            stream=False
        )
        if result.returncode == 0:
            release_info = json.loads(result.stdout)
            print(f"   Status: {release_info.get('info', {}).get('status', 'unknown')}")
            print(f"   Chart: {release_info.get('chart', {}).get('metadata', {}).get('name', 'unknown')}")
            print(f"   Version: {release_info.get('chart', {}).get('metadata', {}).get('version', 'unknown')}")
    else:
        warn(f"Helm release '{helm_release}' not found in namespace '{namespace}'")
        print("\n💡 You need to deploy LangSmith first using Module 1 notebooks.")
        print("   See the 'Deploy Environment' section below.")
        raise RuntimeError("Helm release not found. Deploy using Module 1 first.")
else:
    warn("Could not list Helm releases")
    print("💡 Make sure Helm is installed and kubectl is configured correctly")


## 4. Verify Ingress Endpoint

Check that the LangSmith ingress is configured and reachable.


In [ ]:
import requests
from urllib.parse import urlparse

print("### Checking Ingress\n")
result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

ingress_found = False
ingress_host = None

if result.returncode == 0:
    ingresses = json.loads(result.stdout)
    for ingress in ingresses.get("items", []):
        rules = ingress.get("spec", {}).get("rules", [])
        for rule in rules:
            host = rule.get("host", "")
            if host:
                ingress_found = True
                ingress_host = host
                print(f"   Found ingress with host: {host}")
                break

if not ingress_found:
    warn("No ingress found")
    print("💡 Ingress may still be provisioning. Check Module 1 validation notebook.")
else:
    ok(f"Ingress configured with host: {ingress_host}")
    
    # Try to reach the endpoint
    if config["LANGSMITH_DOMAIN"]:
        test_url = f"https://{config['LANGSMITH_DOMAIN']}"
    elif ingress_host:
        test_url = f"https://{ingress_host}"
    else:
        test_url = None
    
    if test_url:
        print(f"\n### Testing Endpoint Reachability\n")
        print(f"Testing: {test_url}")
        try:
            # Allow redirects, don't verify SSL (may be self-signed)
            response = requests.get(test_url, allow_redirects=True, verify=False, timeout=10)
            if response.status_code in [200, 302, 401, 403]:
                ok(f"Endpoint is reachable (HTTP {response.status_code})")
            else:
                warn(f"Endpoint returned unexpected status: {response.status_code}")
        except requests.exceptions.SSLError:
            # SSL error is OK if using self-signed certs
            warn("SSL verification failed (may be self-signed certificate)")
            print("💡 This is OK for testing. In production, use proper TLS certificates.")
        except requests.exceptions.RequestException as e:
            warn(f"Could not reach endpoint: {e}")
            print("💡 Ingress may still be provisioning. Wait a few minutes and try again.")
    else:
        warn("No domain configured for testing")
        print("💡 Set LANGSMITH_DOMAIN in your .env file to test endpoint reachability")


## 5. Quick Health Check

Verify that key deployments are running.


In [ ]:
print("### Checking Key Deployments\n")
result = run(
    ["kubectl", "get", "deployments", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    deployments = json.loads(result.stdout)
    deployment_items = deployments.get("items", [])
    
    if deployment_items:
        ok(f"Found {len(deployment_items)} deployment(s)")
        print("\nDeployment Status:")
        for deployment in deployment_items:
            name = deployment.get("metadata", {}).get("name", "")
            spec_replicas = deployment.get("spec", {}).get("replicas", 0)
            status_replicas = deployment.get("status", {}).get("replicas", 0)
            ready_replicas = deployment.get("status", {}).get("readyReplicas", 0)
            available_replicas = deployment.get("status", {}).get("availableReplicas", 0)
            
            status_icon = "✅" if ready_replicas == spec_replicas and available_replicas == spec_replicas else "⚠️"
            print(f"   {status_icon} {name}: {ready_replicas}/{spec_replicas} ready, {available_replicas}/{spec_replicas} available")
    else:
        warn("No deployments found")
        print("💡 LangSmith may not be fully deployed. Check Module 1 validation notebook.")
else:
    warn("Could not list deployments")


## ✅ Environment Ready

Your LangSmith environment is running and accessible. You're ready to proceed with Module 4 failure labs.

**Next Steps:**
1. Run `01_diagnostics_baseline.ipynb` to capture a baseline snapshot
2. Proceed with failure labs (10, 20, 30, 40)
3. Optionally run `90_full_incident_drill.ipynb` for a complete incident simulation

---

## 📝 Important Reminder

**When finished with Module 4, run Module 1's `99_teardown.ipynb` to delete the environment and avoid ongoing cloud costs.**

The teardown notebook will:
- Remove Helm release
- Destroy Terraform-managed infrastructure (Kubernetes cluster, database, cache, blob storage, etc.)
- Clean up any remaining resources

**Location:** `../module-1/99_teardown.ipynb`


## 🚀 Deploy Environment (If Not Already Deployed)

If your environment is not running, follow these steps to deploy LangSmith using Module 1:

### Step 1: Preflight Checks
Run `../module-1/01_preflight.ipynb` to validate your environment.

### Step 2: Provision Infrastructure
Run `../module-1/02_terraform_apply.ipynb` to deploy cloud infrastructure (Kubernetes cluster, database, cache, blob storage).

### Step 3: Install LangSmith
Run `../module-1/03_helm_install_langsmith.ipynb` to install LangSmith using Helm.

### Step 4: Validate Deployment
Run `../module-1/04_validate_ingress_and_ui.ipynb` to verify everything is working.

### Step 5: Return Here
Once deployment is complete, return to this notebook and re-run the cells above to verify your environment is ready for Module 4.

---

**Note:** If you encounter errors during deployment, refer to Module 1 documentation and troubleshooting guides.
